In [8]:
import re
from pathlib import Path
from urllib.parse import urljoin, urlparse

import pandas as pd
import requests
import xarray as xr
from bs4 import BeautifulSoup

In [9]:
DATA_PATH = Path("./data")
OUT_PATH = DATA_PATH / "output" / "historical_noaa"
OUT_PATH.mkdir(exist_ok=True)

ROOT_URL = (
    "https://www.ncei.noaa.gov/data/sea-surface-temperature-optimum-interpolation/"
    "v2.1/access/avhrr/"
)
SCRAPED_TABLE_PATH = OUT_PATH / "urls.parquet"

`ROOT_URL` es un directorio de archivos. Cada subdirectorio es un año y mes (por ejemplo, `198109`). Dentro de cada subdirectorio hay un archivo netCDF por día, con la temperatura superficial. En la siguiente celda definimos funciones para iterar sobre este directorio y listar todas las URLs de descarga.

In [10]:
def get_all_urls(root_url: str) -> list[str]:
    response = requests.get(root_url, timeout=60)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")
    urls: list[str] = []
    for link in soup.find_all("a"):
        href = link.get("href")
        if not isinstance(href, str) or "Parent" in link.get_text():
            continue
        url = urljoin(root_url, href)
        if url.startswith(root_url) and url != root_url:
            urls.append(url)
    return list(dict.fromkeys(urls))


def build_url_table(urls: list[str]) -> pd.DataFrame:
    records = []
    for url in urls:
        filename = urlparse(url).path.rsplit("/", 1)[-1]
        if filename.startswith("oisst-avhrr-v02r01.") and filename.endswith(".nc"):
            date_str = filename.split(".")[1]
            if not date_str.endswith("_preliminary"):
                date = pd.to_datetime(date_str, format="%Y%m%d", errors="raise")
                records.append({"url": url, "date": date})
    return pd.DataFrame(records, columns=["url", "date"])


def discover_archive(root_url: str = ROOT_URL) -> pd.DataFrame:
    all_urls: list[str] = []
    for month_url in get_all_urls(root_url):
        directory = urlparse(month_url).path.rstrip("/").rsplit("/", 1)[-1]
        if re.fullmatch(r"\d{6}", directory):
            all_urls.extend(get_all_urls(f"{month_url.rstrip('/')}/"))
    return build_url_table(all_urls)

Al final, obtenemos una con fechas desde 1981 hasta 2025, con la URL para descargar el archivo netCDF de ese día.

In [11]:
if not SCRAPED_TABLE_PATH.exists():
    df_url = discover_archive(ROOT_URL)
    df_url.to_parquet(SCRAPED_TABLE_PATH)
else:
    df_url = pd.read_parquet(SCRAPED_TABLE_PATH)

df_url.head(5)

,url,date
0,https://www.ncei.noaa.gov/data/sea-surface-tem...,1981-09-01
1,https://www.ncei.noaa.gov/data/sea-surface-tem...,1981-09-02
2,https://www.ncei.noaa.gov/data/sea-surface-tem...,1981-09-03
3,https://www.ncei.noaa.gov/data/sea-surface-tem...,1981-09-04
4,https://www.ncei.noaa.gov/data/sea-surface-tem...,1981-09-05


Definimos una función para descargarlos y ordenar sus coordenadas.

In [12]:
def open_netcdf_from_url(url: str) -> xr.DataArray:
    response = requests.get(url, timeout=60)
    response.raise_for_status()
    with xr.open_dataset(response.content) as dataset:
        ds = (
            dataset.assign_coords(lon=lambda x: ((x.lon + 180) % 360) - 180)
            .load()
            .rename(
                {
                    "lon": "x",
                    "lat": "y",
                }
            )
        )

        if len(ds.zlev) > 1 or len(ds.time) > 1:
            err = f"Expected single zlev and time, got {len(ds.zlev)} zlev and {len(ds.time)} time"
            raise ValueError(err)

        ds = (
            ds.squeeze(dim=["zlev", "time"], drop=True)
            .rio.set_spatial_dims(x_dim="x", y_dim="y")
            .rio.write_crs("EPSG:4326")
            .sortby(["x", "y"])
        )
        return ds.sst

Agrupamos las URLs por mes, descargamos todos los archivos para los días de ese mes, y los promediamos para obtener un solo raster para todo el mes. Lo guardamos en formato .tif

In [13]:
urls_by_month = (
    df_url.assign(year_month=lambda df: df["date"].dt.to_period("M"))
    .groupby("year_month")["url"]
    .apply(list)
)

In [14]:
def download_and_average_monthly_data(urls: list[str]) -> xr.DataArray:
    monthly_data = []
    for url in urls:
        ds = open_netcdf_from_url(url)
        monthly_data.append(ds)
    combined = xr.concat(monthly_data, dim="time")
    return combined.mean(dim="time", skipna=True)

In [15]:
for year_month, urls in urls_by_month.items():
    fpath = OUT_PATH / f"{year_month}.tif"
    if fpath.exists():
        print(f"Skipping {year_month} as output file already exists")
        continue

    print(f"Processing {year_month} with {len(urls)} files")
    processed = download_and_average_monthly_data(urls)
    processed.rio.to_raster(OUT_PATH / f"{year_month}.tif", compress="ZSTD")

Skipping 1981-09 as output file already exists
Skipping 1981-10 as output file already exists
Skipping 1981-11 as output file already exists
Skipping 1981-12 as output file already exists
Skipping 1982-01 as output file already exists
Skipping 1982-02 as output file already exists
Skipping 1982-03 as output file already exists
Skipping 1982-04 as output file already exists
Skipping 1982-05 as output file already exists
Processing 1982-06 with 30 files
Skipping 1982-07 as output file already exists
Skipping 1982-08 as output file already exists
Skipping 1982-09 as output file already exists
Skipping 1982-10 as output file already exists
Skipping 1982-11 as output file already exists
Skipping 1982-12 as output file already exists
Skipping 1983-01 as output file already exists
Skipping 1983-02 as output file already exists
Skipping 1983-03 as output file already exists
Skipping 1983-04 as output file already exists
Skipping 1983-05 as output file already exists
Skipping 1983-06 as output 